# Повторный запуск

Нужны Python 3.12, доступ к интернету и GPU с нативной поддержкой bfloat16 и достаточной памятью. Поместите каталог проекта в `/content/lab1_officehome` или `/kaggle/working/lab1_officehome`. Первая ячейка создаёт отдельную рабочую копию кода и конфигураций; изображения загружаются из зафиксированной версии Office-Home. При возобновлении расчёта используйте ранее созданный рабочий каталог и переходите к следующей ячейке.

In [ ]:
from pathlib import Path
import os, subprocess, sys, shutil
SOURCE = (Path('/content') if Path('/content').exists() else Path('/kaggle/working')) / 'lab1_officehome'
assert (SOURCE/'configs/experiment.json').exists()
assert sys.version_info[:2] == (3,12), 'Для зафиксированной среды нужен Python 3.12'
ROOT = SOURCE.with_name('lab1_officehome_reproduction')
assert not ROOT.exists(), 'Каталог уже существует: для возобновления задайте ROOT и пропустите копирование'
ROOT.mkdir()
for name in ['src','configs','literature']:
    shutil.copytree(SOURCE/name,ROOT/name)
(ROOT/'data').mkdir()
shutil.copy2(SOURCE/'data/download_metadata.json',ROOT/'data/download_metadata.json')
for name in ['requirements.txt','README.md','REQUIREMENTS.md','report_background.md','DATA_AND_MODEL_NOTICES.md','Паспорт_данных.md']:
    shutil.copy2(SOURCE/name,ROOT/name)
(ROOT/'logs').mkdir()
(ROOT/'results').mkdir()
os.chdir(ROOT)
subprocess.check_call([sys.executable,'-m','pip','install','-r','requirements.txt'])

In [ ]:
import torch
print(torch.__version__,torch.cuda.is_available())
assert torch.cuda.is_available(), 'Выберите GPU runtime'
print(torch.cuda.get_device_name(0))
assert torch.cuda.is_bf16_supported(including_emulation=False), 'Для текущего протокола Qwen нужен GPU с нативным bfloat16; другой режим точности требует отдельной проверки'
print('GPU bytes:',torch.cuda.get_device_properties(0).total_memory)
# Меньшая память может вызвать OOM: модель few-shot обрабатывает четыре изображения.

In [ ]:
import json
subprocess.check_call([sys.executable,'src/download_data.py'])
subprocess.check_call([sys.executable,'src/verify.py'])
subprocess.check_call([sys.executable,'src/analyze_data.py'])
with Path('configs/environment_freeze.txt').open('w') as out:
    subprocess.check_call([sys.executable,'-m','pip','freeze'],stdout=out)

In [ ]:
cfg=json.loads(Path('configs/experiment.json').read_text())
for spec in cfg['models']:
    subprocess.check_call([sys.executable,'src/download_models.py','--model',spec['key']])
    subprocess.check_call([sys.executable,'src/benchmark.py','--model',spec['key']])
subprocess.check_call([sys.executable,'src/summarize.py'])
selected=json.loads(Path('results/selection.json').read_text())
subprocess.check_call([sys.executable,'src/benchmark.py','--model',selected['model'],'--application'])
subprocess.check_call([sys.executable,'src/adaptation.py'])

In [ ]:
subprocess.check_call([sys.executable,'src/train_lora.py','--probe'])
subprocess.check_call([sys.executable,'src/train_lora.py'])
subprocess.check_call([sys.executable,'src/benchmark.py','--model','smol500','--adapter','results/peft/adapter'])
for script in ['summarize_peft.py','analyze_results.py','verify_results.py']:
    subprocess.check_call([sys.executable,'src/'+script])

Все новые ответы и журналы находятся в `lab1_officehome_reproduction`; исходный комплект не изменяется. Локальные показатели latency нельзя считать облачными. В случае OOM следует выбрать устройство с большей памятью; изменение разрешения или few-shot состава создаёт другой протокол, который нельзя объединять с исходными результатами.